In [1]:
import os
import sys
import time

import numpy as np
import pandas as pd
import anndata as ad

import NaiveDE
import SpatialDE

from datetime import date
today = date.today()

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

Helpers to load data and save results

In [2]:
def load_visium(path) -> ad.AnnData:
    """
    Load a single Visium sample into an AnnData with obsm["spatial"]
    """
    adata = ad.read_h5ad(os.path.join(path, 'adata.h5ad'))
    adata.var_names_make_unique()
    
    return adata

def _unique_dir(path: str) -> str:
    """`path`, or `path` with a numeric suffix (_1, _2, ...) appended to
    its final component if it already exists -- run directories are
    never overwritten."""
    if not os.path.exists(path):
        return path
    parent, name = os.path.split(path)
    i = 1
    while True:
        candidate = os.path.join(parent, f"{name}_{i}")
        if not os.path.exists(candidate):
            return candidate
        i += 1



Load data and make output directory

In [3]:
output_root = '../results'
dataset_id = 'libd'
sample_id = '151673'
run_id = 'spatialde'

In [4]:
base_dir = os.path.join(output_root, dataset_id, sample_id, run_id)
out_dir = _unique_dir(base_dir)
os.makedirs(out_dir)

print(f"Writing results to {out_dir}", flush=True)

Writing results to ../results/libd/151673/spatialde_2


Load same AnnData object that was used to fit the full model

In [5]:
data_path = os.path.join(out_dir, "../stipple/")

In [6]:
adata = load_visium(data_path)

Process data as in the notebooks [here](https://github.com/Teichlab/SpatialDE/tree/master/Analysis)

In [7]:
sample_info = adata.obs
sample_info = sample_info.query('total_counts > 10')  # Remove empty features

X = sample_info[['array_row', 'array_col']].values
df = pd.DataFrame(adata.X.toarray(), columns=adata.var_names, index=adata.obs_names)

Uncomment to run spatialDE, otherwise, load relevant results from ouptut folder

In [8]:
# dfm = NaiveDE.stabilize(df.T).T
# res = NaiveDE.regress_out(sample_info, dfm.T, 'np.log(total_counts)').T
# res['log_total_count'] = np.log(sample_info['total_counts'])

In [9]:
# results = SpatialDE.run(X, res)

# de_results = results[(results.qval < 0.05)].copy()
# de_results.to_csv(os.path.join(out_dir, 'de_results.csv'))

# ms_results = SpatialDE.model_search(X, res, de_results)
# fit_seconds = time.perf_counter() - t0

# ms_results.to_csv(os.path.join(out_dir, 'model_search.csv'))

results = pd.read_csv(os.path.join(out_dir, '../spatialde/de_results.csv'))

In [10]:
sde_genes = results.g.values

In [11]:
adata.var['is_spatialde_gene'] = adata.var.gene.isin(sde_genes)

How does the stipple classification align with SpatialDE 

In [13]:
adata.var.groupby(['classification','is_spatialde_gene']).size()

/tmp/ipykernel_2329293/2333766712.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.var.groupby(['classification','is_spatialde_gene']).size()


classification       is_spatialde_gene
DVG                  False                 210
                     True                 1567
DVG + Field aligned  False                   1
                     True                  100
Field aligned        False                   0
                     True                    1
Null Gene            False                  54
                     True                  182
SVG                  False                   0
                     True                   12
dtype: int64